Referências: https://andersonuyekita.github.io/notebooks/blog/2019/03/21/como-usar-o-gridsearchcv/
https://medium.com/@dicee/the-main-parameters-in-xgboost-and-their-effects-on-model-performance-4f9833cac7c


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

df = pd.read_csv("dataset.csv")

In [2]:
from sklearn.model_selection import GridSearchCV

X = df.drop(columns="victory")
y = df.iloc[:, -1]

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

modelo = XGBClassifier(random_state=42, n_jobs=1)

In [3]:
from sklearn.model_selection import StratifiedShuffleSplit

# Usando o constructor para criar o objeto sss
sss = StratifiedShuffleSplit(n_splits = 5,       # 20 simulações.
                             test_size = 0.2,     # 20% do dataset será de testes.
                             random_state = 42)   # Permitir a reprodutibilidade.


In [4]:
# range de parametros que o grid search vai testar
parametros = {'n_estimators':[100, 300, 500],
             'learning_rate':[0.05, 0.1, 0.5],
              'max_depth': [3, 6],
              'subsample': [0.5, 1],
              'colsample_bytree': [0.5, 1],
              'lambda': [1, 5],
              'alpha': [0, 1]
             }



# Criação do objeto do GridSearchCV.
grid = GridSearchCV(estimator = modelo,          # É o nosso AdaBoost.
                    param_grid = parametros,  # É aquele dicionário com valores para serem testados.
                    scoring = 'f1',           # Arbitrariamente escolhi o f1, adiante explico com detalhes.
                    cv = sss,    # Quantas vezes o dataset passa por cross validation (nosso K)
                    n_jobs = -1) # Usa todos os núcleos do processador.

In [5]:
# Treinando o grid.
grid.fit(X_treino, y_treino)

# Imprimindo os resultados.
pd.DataFrame(grid.cv_results_)

pd.DataFrame(grid.cv_results_)[['params','rank_test_score','mean_test_score']]


,params,rank_test_score,mean_test_score
0,"{'alpha': 0, 'colsample_bytree': 0.5, 'lambda'...",274,0.892429
1,"{'alpha': 0, 'colsample_bytree': 0.5, 'lambda'...",279,0.888034
2,"{'alpha': 0, 'colsample_bytree': 0.5, 'lambda'...",193,0.936633
3,"{'alpha': 0, 'colsample_bytree': 0.5, 'lambda'...",234,0.928855
4,"{'alpha': 0, 'colsample_bytree': 0.5, 'lambda'...",91,0.945353
...,...,...,...
283,"{'alpha': 1, 'colsample_bytree': 1, 'lambda': ...",213,0.934592
284,"{'alpha': 1, 'colsample_bytree': 1, 'lambda': ...",35,0.949518
285,"{'alpha': 1, 'colsample_bytree': 1, 'lambda': ...",128,0.942901
286,"{'alpha': 1, 'colsample_bytree': 1, 'lambda': ...",7,0.951825


In [6]:
# Imprime os parâmetros que produziram o ".best_score_".
print(grid.best_params_)

# Imprimindo o score.
print(grid.best_score_)

{'alpha': 0, 'colsample_bytree': 1, 'lambda': 1, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.5}
0.9543873768782133
